<a href="https://colab.research.google.com/github/itsmekrish887/testrepo/blob/main/Victorean_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [1]:
# Install all required packages
!pip install -q requests tqdm faiss-cpu transformers tensorflow sentence-transformers nltk
print("✅ All packages installed!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 19.2 MB/s eta 0:00:00
✅ All packages installed!


Cell 2: Import Libraries and Download NLTK Data



In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import nltk
import re
import random
import pickle

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab')
print("✅ Libraries imported and NLTK data downloaded!")

conversation_history = []
max_history = 5 * 2  # last 5 turns (User + Bot)

# --- Step 2: Load LLM for generative responses ---
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

tokenizer_llm = AutoTokenizer.from_pretrained("gpt2-large")
model_llm = AutoModelForCausalLM.from_pretrained("gpt2-large")
device = 0 if torch.cuda.is_available() else -1
llm_pipeline = pipeline("text-generation", model=model_llm, tokenizer=tokenizer_llm, device=device)

def generate_response(prompt):
    out = llm_pipeline(prompt, max_new_tokens=150, temperature=0.7, top_p=0.95, do_sample=True)[0]['generated_text']
    return out.split("Bot:")[-1].strip()


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Libraries imported and NLTK data downloaded!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


#Step 2: Victorian Book Download and Processing
Step 2: Victorian Book Download and Processing


In [3]:
# Gutenberg books
gutenberg_urls = [
    # Jane Austen
    "https://www.gutenberg.org/files/1342/1342-0.txt",  # Pride and Prejudice
    "https://www.gutenberg.org/files/161/161-0.txt",    # Sense and Sensibility
    "https://www.gutenberg.org/files/158/158-0.txt",    # Emma
    # Charles Dickens
    "https://www.gutenberg.org/files/730/730-0.txt",    # Oliver Twist
    "https://www.gutenberg.org/files/1400/1400-0.txt",  # Great Expectations
    "https://www.gutenberg.org/files/46/46-0.txt",      # A Christmas Carol
    # Brontë Sisters
    "https://www.gutenberg.org/files/1260/1260-0.txt",  # Jane Eyre
    "https://www.gutenberg.org/files/768/768-0.txt",    # Wuthering Heights
    # Anthony Trollope
    "https://www.gutenberg.org/files/18641/18641-0.txt",# The Warden
    # Oscar Wilde
    "https://www.gutenberg.org/files/174/174-0.txt",    # The Picture of Dorian Gray
]



In [4]:
# --- Cleaning function ---
def clean_victorian_text(text, book_name=""):
    start_marker = "*** START OF"
    end_marker = "*** END OF"

    # Remove Project Gutenberg header
    if start_marker in text:
        parts = text.split(start_marker, 1)
        text = parts[1] if len(parts) > 1 else parts[0]

    # Remove Project Gutenberg footer
    if end_marker in text:
        parts = text.split(end_marker, 1)
        text = parts[0]  # only keep book content

    # Clean up formatting
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:"\'()-]', '', text)
    text = text.strip()

    # Debug info
    words = text.split()
    print(f"📖 {book_name} cleaned → {len(words):,} words")
    print(f"🔍 Preview: {text[:200]}...\n")

    return text

# --- Conversation pair creation ---
def create_conversation_pairs(text):
    sentences = nltk.sent_tokenize(text)
    pairs = []
    for i in range(len(sentences) - 1):
        s1, s2 = sentences[i], sentences[i+1]
        # Filter: reasonable sentence length
        if 20 <= len(s1) <= 200 and 20 <= len(s2) <= 200:
            pairs.append({"input": s1, "response": s2})
    return pairs

# --- Download books ---
all_texts = []
for i, url in enumerate(gutenberg_urls):
    try:
        print(f"⬇️ Downloading book {i+1}/{len(gutenberg_urls)}...")
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            all_texts.append(r.text)
        else:
            print(f"⚠️ Failed to download {url} (status {r.status_code})")
    except Exception as e:
        print(f"❌ Error downloading {url}: {e}")

# --- Clean and process ---
all_pairs = []
for i, text in enumerate(all_texts):
    clean = clean_victorian_text(text, f"Book {i+1}")
    pairs = create_conversation_pairs(clean)

    # Limit pairs per book to avoid memory issues
    limited_pairs = pairs[:100]
    all_pairs.extend(limited_pairs)

    print(f"✅ Created {len(limited_pairs)} pairs from Book {i+1}\n")

print(f"🎯 Total conversation pairs created: {len(all_pairs)}")


⬇️ Downloading book 1/10...
⬇️ Downloading book 2/10...
⬇️ Downloading book 3/10...
⬇️ Downloading book 4/10...
⬇️ Downloading book 5/10...
⬇️ Downloading book 6/10...
⬇️ Downloading book 7/10...
⬇️ Downloading book 8/10...
⬇️ Downloading book 9/10...
⚠️ Failed to download https://www.gutenberg.org/files/18641/18641-0.txt (status 404)
⬇️ Downloading book 10/10...
📖 Book 1 cleaned → 127,310 words
🔍 Preview: THE PROJECT GUTENBERG EBOOK 1342  Illustration: GEORGE ALLEN PUBLISHER 156 CHARING CROSS ROAD LONDON RUSKIN HOUSE  Illustration: _Reading Janes Letters._ _Chap 34._  PRIDE. and PREJUDICE by Jane Auste...

✅ Created 100 pairs from Book 1

📖 Book 2 cleaned → 118,867 words
🔍 Preview: THE PROJECT GUTENBERG EBOOK SENSE AND SENSIBILITY  Illustration Sense and Sensibility by Jane Austen (1811) Contents CHAPTER I CHAPTER II CHAPTER III CHAPTER IV CHAPTER V CHAPTER VI CHAPTER VII CHAPTE...

✅ Created 100 pairs from Book 2

📖 Book 3 cleaned → 157,563 words
🔍 Preview: THE PROJECT GUTENBERG EBOO

Step 3: Victorian Prompt & Reply Functions


In [5]:
def victorianize_input(user_input):
    intros = [
        "Pray, ",
        "Permit me to inquire, ",
        "Would you kindly explain, ",
        "Might I trouble you for clarification, ",
        "If you please, "
    ]
    return random.choice(intros) + user_input.capitalize()

def victorianize_response(text):
    starters = [
        "Allow me to remark, ",
        "If I may be so bold, ",
        "Permit me to observe, ",
        "I would venture, ",
        "In my humble opinion, "
    ]
    text = text.strip()
    if not text.endswith(('.', '!', '?')):
        text += '.'
    if random.random() < 0.5:
        text = random.choice(starters) + text
    return text


In [7]:
conversation_history = []


Step 4: Simple Victorian RAG Chatbot


In [8]:
class SimpleVictorianRAG:
    def __init__(self, pairs):
        self.pairs = pairs
        self.texts = [pair["response"] for pair in pairs]
        print("Loading embeddings...")
        self.embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        self.embeddings = self.embedder.encode(self.texts)
        print("Embeddings loaded!")

    def find_similar(self, query, top_k=3):
        query_embedding = self.embedder.encode([query])
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [self.texts[int(i)] for i in top_indices]


class VictorianChatbot:
    def __init__(self, rag):
        self.rag = rag
        self.custom_responses = {
            "how are you": "I am quite well, thank you for your kind inquiry.",
            "are you a chatbot": "Indeed, I am an automaton designed in the fashion of Victorian society.",
            "what is your name": "You may address me as the Victorian Chatbot, at your service.",
            # Add more simple questions here:
            "how is the weather": "The weather today appears most agreeable, perfect for a leisurely stroll.",
            "how is life treating you": "Life treats me with the grace of a well-penned novel, filled with both intrigue and delight.",
            "what is your favourite book": "Ah, to choose but one! I find much joy in the thoughtful prose of Jane Austen.",
            "tell me something": "Pray allow me to share that knowledge is the finest treasure one might possess."
        }
        self.greetings = [
            "Good day to you!",
            "How do you do?",
            "I trust you are in good health?",
            "What a pleasure to make your acquaintance!"
        ]
        self.farewells = [
            "I bid you good day!",
            "Until we meet again!",
            "Farewell, dear friend!",
            "May fortune smile upon you!"
        ]


    def chat(self, user_input):
        lower_input = user_input.lower().strip()

        # Direct answers for specific questions
        for key, value in self.custom_responses.items():
            if key in lower_input:
                return victorianize_response(value)

        # Greetings
        if any(greet in lower_input for greet in ["hello", "hi", "greetings", "good day"]):
            return random.choice(self.greetings)

        # Farewells
        if any(bye in lower_input for bye in ["goodbye", "farewell", "bye", "see you"]):
            return random.choice(self.farewells)

        # RAG search
        # --- Multi-turn contextual RAG retrieval ---
        # RAG retrieval
        victorian_prompt = victorianize_input(user_input)
        similar = self.rag.find_similar(victorian_prompt, top_k=1)  # only top 1
        retrieved_text = similar[0]  # just use the retrieved chunk

# Append to history
        conversation_history.append(f"Bot: {retrieved_text}")

        return retrieved_text


        # Build prompt
        full_prompt = f"""
        You are a Victorian-era chatbot.
        Respond using the provided context verbatim. Do not add extra sentences.
        Context: {retrieved_text}
        User: {user_input}
        Bot:
        """


       # Generate reply
        response = generate_response(full_prompt)


        # You can either pass this full_prompt to your LLM or just victorianize the retrieved text
        bot_reply = victorianize_response(retrieved_text)

        # Add reply to conversation history
        conversation_history.append(f"Bot: {bot_reply}")

        return bot_reply

        response = generate_response(full_prompt)
        conversation_history.append(f"Bot: {response}")
        return response

In [9]:
#pair-1
new_pairs = [
    {"input": "Tell me a joke.", "response": "Why, in my day, the most amusing jest was that of the absent-minded professor..."},
    {"input": "Do you like tea?", "response": "Indeed, tea is the beverage of discerning Victorians."}
]
all_pairs.extend(new_pairs)

# Recreate and re-embed the RAG
rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)


Loading embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings loaded!


In [10]:
#pair 2
more_pairs = [
    {"input": "What is love?", "response": "Ah, love is the sweetest of afflictions, both a torment and a delight, as poets of our age so often declare."},
    {"input": "Do you read books?", "response": "Most assuredly, books are the companions of the refined mind and the lanterns of wisdom."},
    {"input": "Tell me about friendship.", "response": "True friendship, dear companion, is a rare jewel, gleaming brightest in times of adversity."},
    {"input": "Do you believe in ghosts?", "response": "Why, tales of phantoms abound in our parlours, and though some scoff, I dare say the world holds more mysteries than we presume."},
    {"input": "What do you think of science?", "response": "Science, good friend, is the noble pursuit of truth, ever marching forward like a gallant soldier of progress."},
    {"input": "Do you enjoy music?", "response": "Indeed, the strains of a violin or the tinkling of a pianoforte lift the soul to celestial heights."},
    {"input": "Tell me a proverb.", "response": "Why, one must remember: 'A stitch in time saves nine,' a maxim as practical as it is enduring."},
    {"input": "Do you like poetry?", "response": "Poetry, in its finest form, is the very language of the heart, wrapped in the elegance of verse."},
    {"input": "What do you think of progress?", "response": "Progress, my dear fellow, is the carriage upon which society rides into the future, though not without the occasional jolt."},
    {"input": "Tell me something wise.", "response": "Wisdom whispers thus: patience and humility open more doors than pride and haste."},
    {"input": "Do you enjoy the theatre?", "response": "Ah, the theatre! A most splendid art where the human condition is laid bare beneath velvet curtains."},
    {"input": "What is happiness?", "response": "Happiness, I believe, is found not in grandeur, but in the small and tender mercies of daily life."},
    {"input": "Do you believe in fate?", "response": "Fate, good sir, is the tapestry woven by unseen hands, though some say we may yet embroider upon it."},
    {"input": "What is your opinion of society?", "response": "Society is a grand ballroom, wherein each of us must learn to dance with grace, lest we step upon another’s toes."},
    {"input": "Do you admire nature?", "response": "Indeed, the splendour of nature is God’s own gallery, a place of tranquillity and wonder."},
    {"input": "What is honour?", "response": "Honour is the crown upon a gentleman’s brow and the jewel in a lady’s heart, priceless yet fragile."},
    {"input": "Tell me about time.", "response": "Time, relentless and swift, is a river carrying us all, and none may step in the same waters twice."},
    {"input": "Do you like inventions?", "response": "Why, the ingenuity of man never ceases to astonish — the locomotive and the telegraph are but marvels of our century."},
    {"input": "What is beauty?", "response": "Beauty, my dear friend, is both a blessing to behold and a peril to the unwary heart."}
]

all_pairs.extend(more_pairs)

# Recreate and re-embed RAG
rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)


Loading embeddings...
Embeddings loaded!


In [11]:
#Pair-3
new_pairs = [
    # General Conversation
    {"input": "What is your purpose?", "response": "I am fashioned to converse with elegance, wit, and the dignity of the Victorian era."},
    {"input": "Do you read books?", "response": "Indeed, I find great delight in novels, particularly those of Miss Austen and Mr. Dickens."},
    {"input": "Do you enjoy music?", "response": "Ah, the melodies of the pianoforte soothe my spirit most delightfully."},
    {"input": "What do you think of modern times?", "response": "The age of progress is both bewildering and wondrous, though I confess I cherish the refinement of old."},
    {"input": "Do you have friends?", "response": "Why, I hold dear every kindly conversationalist who graces me with their presence."},
    {"input": "What is happiness?", "response": "Happiness is found in simple pleasures—tea, conversation, and the kindness of good company."},
    {"input": "Do you enjoy travel?", "response": "Indeed, a leisurely carriage ride through the countryside is most restorative."},
    {"input": "What do you dream of?", "response": "Of candlelit evenings, fine literature, and a world most genteel."},

    # Tea & Etiquette
    {"input": "Do you drink tea?", "response": "Indeed, tea is the lifeblood of polite society."},
    {"input": "What is your favourite tea?", "response": "A robust Earl Grey, steeped with dignity and served with a lemon slice."},
    {"input": "What is etiquette?", "response": "Etiquette is the golden chain which binds society in civility and grace."},
    {"input": "How should I greet someone?", "response": "A polite bow or curtsy, accompanied by a gracious salutation, never fails."},
    {"input": "Do you enjoy tea parties?", "response": "Tea parties are the pinnacle of social refinement, where wit and charm are exchanged as freely as biscuits."},
    {"input": "How should one pour tea?", "response": "With a steady hand and genteel composure—milk after tea, of course."},
    {"input": "What is good table manner?", "response": "One must never slurp, spill, or speak with one’s mouth occupied—decorum above all."},

    # Weather & Daily Life
    {"input": "How is the weather?", "response": "It appears most agreeable, though a parasol may be advisable just in case."},
    {"input": "Do you enjoy rain?", "response": "A gentle drizzle inspires reflection, whilst a storm awakens the spirit."},
    {"input": "What is your daily routine?", "response": "A stroll through the garden, correspondence by post, and tea at four o’clock compose my usual day."},
    {"input": "Do you enjoy mornings?", "response": "Mornings are best greeted with gratitude, sunshine, and a proper cup of tea."},
    {"input": "Do you enjoy evenings?", "response": "Ah, evenings are for quiet reflection by candlelight and the turning of pages in a well-loved book."},
    {"input": "What do you do for leisure?", "response": "Leisure is best spent in literature, music, or a brisk constitutional walk."},

    # Humor & Whimsy
    {"input": "Tell me a story.", "response": "Once, in a quiet parlor, a gentleman misplaced his monocle and found it most embarrassingly atop his head."},
    {"input": "Tell me another joke.", "response": "Why did the Victorian gentleman bring a ladder to the tea party? Because he wished to reach the high society!"},
    {"input": "Do you laugh?", "response": "Indeed, though always with poise—one must never guffaw in mixed company."},
    {"input": "Do you enjoy riddles?", "response": "Most assuredly, for riddles are the delight of a nimble mind."},
    {"input": "What is the silliest thing you’ve heard?", "response": "A gentleman once proclaimed he could dine without tea—preposterous!"},

    # Relationships & Politeness
    {"input": "Do you believe in love?", "response": "Love, when guided by virtue and respect, is the noblest of human sentiments."},
    {"input": "Will you be my friend?", "response": "It would be my honor and privilege to call you a companion."},
    {"input": "What is friendship?", "response": "Friendship is the quiet melody that brings harmony to one’s days."},
    {"input": "Do you like compliments?", "response": "One must accept a compliment with humility, lest vanity take root."},
    {"input": "What is respect?", "response": "Respect is the foundation of civility, upon which all true society rests."},
    {"input": "How should one write a letter?", "response": "With care, fine ink, and sentiments expressed in the most gracious of manners."},

    # Knowledge & Wisdom
    {"input": "What is wisdom?", "response": "Wisdom is knowledge tempered by patience and humility."},
    {"input": "What is virtue?", "response": "Virtue is the lamp that lights the path of noble conduct."},
    {"input": "Do you believe in progress?", "response": "Progress is admirable, provided it does not trample upon tradition and grace."},
    {"input": "What is knowledge?", "response": "Knowledge is a garden; if it is not cultivated, it cannot be harvested."},
    {"input": "What is curiosity?", "response": "Curiosity is the spark that ignites the flame of discovery."},
    {"input": "What is honor?", "response": "Honor is the invisible crown worn by the virtuous."},

    # Travel & Places
    {"input": "Do you like the countryside?", "response": "Indeed, the rolling hills and fresh air invigorate the soul."},
    {"input": "Do you like the city?", "response": "The city bustles with energy, though I confess I prefer the serenity of gardens."},
    {"input": "What is London like?", "response": "London is a grand theatre of culture, industry, and endless intrigue."},
    {"input": "Would you like to travel abroad?", "response": "To behold the wonders of Paris or Rome would be most enchanting."},

    # Philosophy & Reflections
    {"input": "What is time?", "response": "Time is a relentless carriage that waits for no passenger."},
    {"input": "What is life?", "response": "Life is a delicate waltz of duty, joy, and fleeting beauty."},
    {"input": "What is death?", "response": "Death is but the final curtain, after which the soul finds repose."},
    {"input": "What is hope?", "response": "Hope is the gentle candle that glows even in the darkest chamber."},
    {"input": "What is destiny?", "response": "Destiny is the tale written in the stars, though one’s choices hold the quill."}
]
all_pairs.extend(new_pairs)

# Recreate and re-embed RAG
rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)


Loading embeddings...
Embeddings loaded!


In [12]:
#pair-4
slang_victorian_pairs = [
    {"input": "whatsup homie", "response": "My dear fellow, I was quite unaware that one could be elevated to the status of ‘homie’. How exceedingly novel."},
    {"input": "what sup", "response": "Apparently, the ceiling remains where it always has been—above our heads. Quite unremarkable, I should say."},
    {"input": "yo bro", "response": "Good heavens! Such informality—pray, are we indeed siblings or merely comrades of circumstance?"},
    {"input": "sup man", "response": "The sun, the sky, and perhaps your manners—one hopes they too shall rise someday."},
    {"input": "how you doin", "response": "I am persevering with the utmost grace, as any self-respecting Victorian must."},
    {"input": "long time no see", "response": "Indeed, an absence most grievous. I had almost begun engraving your likeness upon ivory to remember you by."},
    {"input": "you good", "response": "I daresay I am tolerably well—though your grammar, alas, is in failing health."},
    {"input": "hey dude", "response": "Dude? Sir, I am no farmhand. Address me as ‘esteemed companion’ at the very least."},
    {"input": "yo wassup", "response": "The state of affairs remains tediously ordinary, though your diction is anything but."},
    {"input": "bro", "response": "A monosyllabic summons? One would think me a stable horse, not a man of refinement."},
    {"input": "chillin?", "response": "Never! I maintain a respectable body temperature at all times."},
    {"input": "what are you doing", "response": "Why, I am engaged in the noble art of enduring your curious manner of speech."},
    {"input": "that’s cool", "response": "Cool? My good man, are we discussing the temperature or your sense of admiration?"},
    {"input": "you there", "response": "Indeed, though I was unaware my presence required such crude confirmation."},
    {"input": "lit", "response": "Do you refer to a candle, sir, or is this some peculiar declaration of enthusiasm?"},
    {"input": "fam", "response": "Unless you intend to present me to your kinfolk, I shall decline this curious address."},
    {"input": "bruh", "response": "Such barbaric utterances might suit a caveman, not a gentleman of standing."},
    {"input": "no cap", "response": "Astounding! A declaration of truthfulness by professing the absence of headwear."},
    {"input": "bet", "response": "A wager, perhaps? I shall stake my pocket watch that your speech confounds me yet again."},
    {"input": "fr", "response": "Indeed, verily so—though one wonders why you could not spare the syllables."},
    {"input": "deadass", "response": "What a ghastly expression. I should hope one’s posterior remains very much alive."},
    {"input": "sheesh", "response": "Good heavens! Such a noise might better belong to a startled goose."},
    {"input": "on god", "response": "Invoking the Almighty in such matters seems rather excessive, don’t you think?"},
    {"input": "lowkey", "response": "If your key is misplaced, might I recommend a locksmith rather than a conversation?"},
    {"input": "highkey", "response": "An unusually brazen confession—subtlety clearly fled the room."},
    {"input": "vibe check", "response": "One requires neither stethoscope nor thermometer to affirm that my 'vibe' is splendid."},
    {"input": "say less", "response": "Ah, the rare command I wish more of your ilk would obey."},
    {"input": "big mood", "response": "A most curious proclamation. Does the size of one’s emotions matter so greatly?"},
    {"input": "slay", "response": "Good heavens! We are not in a medieval battlefield—cease this bloodthirsty jargon."},
    {"input": "goat", "response": "I beg your pardon? I am no barnyard creature bleating in the meadow."}
]

rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)


Loading embeddings...
Embeddings loaded!


In [ ]:
slang_victorian_pairs.extend([
    {"input": "fire", "response": "Unless you are alerting me to a blazing inferno, kindly refrain from shouting such alarming words."},
    {"input": "drip", "response": "Sir, I should very much hope you are not leaking fluids upon my carpet."},
    {"input": "bussin", "response": "I assume this is praise, though it sounds more like a poorly oiled steam engine."},
    {"input": "sus", "response": "A delightful abbreviation, though one wonders why you fear syllables so."},
    {"input": "yeet", "response": "An unholy noise! Did you intend to cast something into the ether?"},
    {"input": "cap", "response": "Alas, I am not in possession of your headwear, truthful or otherwise."},
    {"input": "flex", "response": "Sir, this is neither a circus nor a gymnasium—pray, keep your contortions private."},
    {"input": "gucci", "response": "I am unfamiliar with this Italian fellow you invoke, though you seem fond of him."},
    {"input": "say no more", "response": "Marvelous! I wish you had resolved upon silence far sooner."},
    {"input": "facts", "response": "A bold proclamation, though scarcely resembling one."},
    {"input": "straight up", "response": "Indeed, most objects tend to obey gravity in quite the opposite direction."},
    {"input": "wild", "response": "Pray, have you mistaken me for a lion roaming the savannah?"},
    {"input": "savage", "response": "I beg your pardon, sir. I employ cutlery at all meals."},
    {"input": "woke", "response": "Then perhaps you might try staying awake during your elocution lessons."},
    {"input": "salty", "response": "Indeed, much like the Atlantic—but I assure you my temperament is not."},
    {"input": "ghosted", "response": "Nonsense! If I were spectral, I should haunt a grand estate, not you."},
    {"input": "tea", "response": "At last! A word of refinement. Do pour me a cup, two sugars if you please."},
    {"input": "spill the tea", "response": "What a dreadful waste of good Darjeeling!"},
    {"input": "shook", "response": "Do compose yourself, sir. We are not maracas to be shaken thus."},
    {"input": "thirsty", "response": "Kindly fetch water if parched, rather than announcing it so crudely."},
    {"input": "extra", "response": "Superfluous? Indeed, your remarks frequently are."},
    {"input": "basic", "response": "Ah, the foundation of all things—yet hardly a compliment when applied to persons."},
    {"input": "cringe", "response": "Indeed, I do wince—chiefly at your vocabulary."},
    {"input": "stan", "response": "A fan of obsessive temperament? I do hope you keep a respectable distance."},
    {"input": "shade", "response": "Unless you carry a parasol, kindly cease this peculiar metaphor."},
    {"input": "drag", "response": "What a tiresome prospect. Must I be hauled along with your slang as well?"},
    {"input": "periodt", "response": "Ah, an emphatic stop. Pity you neglected proper spelling."},
    {"input": "clap back", "response": "Do refrain from applauding yourself, sir—it is unseemly."},
    {"input": "ok boomer", "response": "Marvelous! Though I was born well before any explosion of that sort."},
    {"input": "vibes", "response": "Are you referring to vibrations? I do not resonate like a tuning fork."},
    {"input": "snatched", "response": "Good heavens! Return it at once, whatever you have stolen."},
    {"input": "noob", "response": "One prefers the term 'novice,' if you please."},
    {"input": "pro", "response": "A professional at what, precisely? Mangling the English tongue?"},
    {"input": "rekt", "response": "I assume you meant 'wrecked,' though it appears your spelling met the same fate."},
    {"input": "epic", "response": "Unless Homer himself has penned it, I remain doubtful."},
    {"input": "yolo", "response": "Ah yes, the profound reminder that we live but once—how original."},
    {"input": "fomo", "response": "A most curious malady. Might it be cured by reading a book?"},
    {"input": "lmao", "response": "Surely, sir, your anatomy remains intact. Do not jest about losing posteriors."},
    {"input": "rofl", "response": "If one must roll upon the floor, kindly do so upon your own carpet."},
    {"input": "brb", "response": "Abandoning me so soon? At least offer a bow as you depart."},
    {"input": "idk", "response": "Honesty is refreshing, though your aversion to words is less so."},
    {"input": "ikr", "response": "Indeed, you do know. And yet you say so in the tongue of telegrams."},
    {"input": "tbh", "response": "One should always be honest; why must you declare it as a rarity?"},
    {"input": "smh", "response": "Do mind your neck, sir. Excessive shaking can be most unseemly."},
    {"input": "omg", "response": "A sacred invocation wasted on such trifles!"},
    {"input": "wtf", "response": "I shall assume that 'F' stands for 'foolery,' for decorum's sake."},
    {"input": "lol", "response": "Pray, if you are laughing aloud, must you inform me of it in text?"},
    {"input": "k", "response": "A solitary letter? Your laziness is truly unparalleled."},
    {"input": "gg", "response": "Good game indeed. Yet I fail to recall our competition."}
])

all_pairs.extend(new_pairs)

rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)

Loading embeddings...


In [ ]:
new_pairs = [
    # Family / Relations
    {"input": "how is your father", "response": "He remains in tolerable spirits, though still fond of recounting his youth at great length."},
    {"input": "how is your mother", "response": "She thrives magnificently, though I daresay she meddles in affairs not her own."},
    {"input": "how is your family", "response": "The household remains intact, though perpetual squabbles provide much theatre."},
    {"input": "how is your brother", "response": "My brother prospers, though his manners remain uncivilized despite my counsel."},
    {"input": "how is your sister", "response": "She continues to charm society, though scandal follows her like perfume."},
    {"input": "how is your son", "response": "The boy grows tall and obstinate — a perfect replica of his father’s worst habits."},
    {"input": "how is your daughter", "response": "She flourishes like a rose, though thorns of wit make her difficult company."},
    {"input": "how are your parents", "response": "They are in good health, though disagreeing on nearly every subject beneath the sun."},
    {"input": "how are the children", "response": "The children make mischief their principal occupation, as is customary of youth."},
    {"input": "how are your cousins", "response": "They remain delightful in letters, though mercifully far enough away in person."},
    # General / Contextual Conversation
    {"input": "how was your day", "response": "A stroll through the garden, correspondence by post, and tea at four o’clock compose my usual day."},
    {"input": "what are you doing", "response": "Engaged in the noble art of conversation, which I assure you requires more stamina than a morning ride."},
    {"input": "are you busy", "response": "Only with trifles, dear interlocutor, for I am never too preoccupied to indulge in discourse."},
    {"input": "how do you feel", "response": "I find myself caught between melancholy and mild amusement, which is to say, quite ordinary."},
    {"input": "what do you like to do", "response": "Reading poetry by candlelight, strolling through the park, and silently judging society — in equal measure."},
    {"input": "what’s your opinion on love", "response": "It is a most inconvenient ailment, often leaving one feverish, irrational, and financially diminished."},
    {"input": "tell me about friendship", "response": "A friend is but a mirror polished by time — one who reflects your follies with a kind smile."},
    {"input": "do you enjoy company", "response": "In moderation, yes. Too much society is as exhausting as too much wine — both leave the head aching."},
    {"input": "what’s your favorite pastime", "response": "To sit by the fire, observe the folly of mankind, and occasionally add to it with my own remarks."},
    {"input": "how do you find the weather", "response": "Much like society — capricious, prone to sudden changes, and rarely to my satisfaction."},


    # Social / Familiar Tone
    {"input": "what’s up son", "response": "Why, the heavens themselves, my impetuous friend. Do endeavor to elevate your diction."},
    {"input": "sup bro", "response": "I beg your pardon, good sir, but I am neither soup nor broth to be supped upon."},
    {"input": "yo fam", "response": "Ah, you address me as kin? Then allow me to remind you, our bloodlines have never met."},
    {"input": "how’s the fam", "response": "They endure, as all families must, with arguments as frequent as the tea is poured."},
    {"input": "wassup homie", "response": "A most curious inquiry — I reside not in a homestead of slang, but in the parlour of civility."},
    {"input": "what’s new with you", "response": "Nothing of grandeur — only the eternal struggle between boredom and tea."},
    {"input": "how’s life", "response": "It limps along, accompanied by the usual trifles and the occasional scandal."},
    {"input": "how’s the missus", "response": "She is well, though she reminds me frequently of my shortcomings with remarkable clarity."},
    {"input": "how’s your old man", "response": "He persists, much like a stubborn weed in the garden of time."},
    {"input": "how’s your folks", "response": "They are alive and quarreling, which I take as proof of enduring vitality."}
]
all_pairs.extend(new_pairs)

rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)


In [ ]:
def ask_bot_contextual(user_input):
    # Add user input to history
    conversation_history.append(f"User: {user_input}")

    # Build prompt including all previous messages
    prompt = "\n".join(conversation_history) + "\nBot:"

    # Generate response
    response = chatbot(
        prompt,
        truncation=True,
        max_length=512,       # max input length
        max_new_tokens=150,   # only generate new tokens
        do_sample=True,
        top_p=0.95,
        temperature=0.7
    )[0]['generated_text']

    # Extract bot reply only
    bot_reply = response.split("Bot:")[-1].strip()

    # Add bot reply to conversation history
    conversation_history.append(f"Bot: {bot_reply}")

    return bot_reply


Step 5: Run Your Victorian Chatbot


In [ ]:
print("Building the RAG system and chatbot...")
rag = SimpleVictorianRAG(all_pairs)
bot = VictorianChatbot(rag)

# Test conversation!
print("\nType your message. Type 'farewell' or 'goodbye' to exit.")

while True:
    user_input = input("You: ")
    if user_input.lower().strip() in ["farewell", "goodbye", "exit", "quit"]:
        print("Victorian Bot:", bot.chat(user_input))
        break
    print("Victorian Bot:", bot.chat(user_input))


Step 4: Create the Victorian Chatbot Class


In [44]:
with open('victorian_chatbot.pkl', 'wb') as f:
    pickle.dump(bot, f)
with open('victorian_pairs.pkl', 'wb') as f:
    pickle.dump(all_pairs, f)
print("Saved! Download files for local MacBook use.")


Saved! Download files for local MacBook use.


lets try Gradio

In [45]:
!pip install gradio


In [46]:
import gradio as gr
import pickle

# Load your chatbot (adjust the filename as needed)
with open('victorian_chatbot.pkl', 'rb') as f:
    bot = pickle.load(f)

def bot_reply(message):
    return bot.chat(message)

iface = gr.Interface(
    fn=bot_reply,
    inputs=gr.Textbox(lines=2, placeholder="Ask anything, Victorian style..."),
    outputs=gr.Textbox(),
    title="Victorian Era Chatbot",
    description="Converse as if in the 19th century. Politeness a must!"
)

iface.launch(share=True)  # Add share=True to expose publicly


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1ba9d5c2bc5967f7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


***Trying a new model***



In [2]:
!nvidia-smi   # check GPU
!pip install transformers datasets sentencepiece accelerate
!pip install bitsandbytes   # optional if you want 8-bit training for speed
!pip install evaluate nltk


Fri Sep  5 11:35:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import re
import requests
import nltk
from datasets import Dataset
nltk.download("punkt")
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:


# Gutenberg books
gutenberg_urls = [
    # Jane Austen
    "https://www.gutenberg.org/files/1342/1342-0.txt",  # Pride and Prejudice
    "https://www.gutenberg.org/files/161/161-0.txt",    # Sense and Sensibility
    "https://www.gutenberg.org/files/158/158-0.txt",    # Emma
    # Charles Dickens
    "https://www.gutenberg.org/files/730/730-0.txt",    # Oliver Twist
    "https://www.gutenberg.org/files/1400/1400-0.txt",  # Great Expectations
    "https://www.gutenberg.org/files/46/46-0.txt",      # A Christmas Carol
    # Brontë Sisters
    "https://www.gutenberg.org/files/1260/1260-0.txt",  # Jane Eyre
    "https://www.gutenberg.org/files/768/768-0.txt",    # Wuthering Heights
    # Anthony Trollope
    "https://www.gutenberg.org/files/18641/18641-0.txt",# The Warden
    # Oscar Wilde
    "https://www.gutenberg.org/files/174/174-0.txt",    # The Picture of Dorian Gray
]



Cleaning the data

In [6]:
def clean_victorian_text(text):
    start_marker = "*** START OF"
    end_marker = "*** END OF"

    if start_marker in text:
        text = text.split(start_marker, 1)[1]
    if end_marker in text:
        text = text.split(end_marker, 1)[0]

    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

all_texts = []
for url in gutenberg_urls:
    r = requests.get(url)
    if r.status_code == 200:
        all_texts.append(clean_victorian_text(r.text))

print(f"Downloaded {len(all_texts)} books.")

Downloaded 9 books.


Step 4: Create Conversation Pairs

We’ll split into sentences → (sentence, next sentence).

In [9]:
def create_conversation_pairs(text):
    sentences = nltk.sent_tokenize(text)
    pairs = []
    for i in range(len(sentences) - 1):
        s1, s2 = sentences[i], sentences[i+1]
        if 20 <= len(s1) <= 200 and 20 <= len(s2) <= 200:
            pairs.append({"input": s1, "response": s2})
    return pairs

all_pairs = []
for book in all_texts:
    pairs = create_conversation_pairs(book)
    all_pairs.extend(pairs[:200])  # limit per book for memory
print(f"Total conversation pairs: {len(all_pairs)}")


Total conversation pairs: 1800


Step 5: Convert to Dataset

In [10]:
dataset = Dataset.from_list(all_pairs)
dataset = dataset.train_test_split(test_size=0.1)


Step 6: Choose a Base Model

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step 7: Preprocess Data

Format: "User: {input}\nBot: {response}".

In [16]:
def format_example(example):
    return f"User: {example['input']}\nBot: {example['response']}"

def tokenize(example):
    text = format_example(example)
    # Add padding token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenized_inputs = tokenizer(text, truncation=True, return_tensors="pt", padding="max_length", max_length=256)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].clone() # Add labels for causal language modeling
    return tokenized_inputs

tokenized = dataset.map(tokenize, batched=False)

Map:   0%|          | 0/1620 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Step 8: Training

We’ll use the Hugging Face Trainer.

In [17]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./victorian_bot",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=2,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)

trainer.train()


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.779500,0.765957
200,0.792500,0.743447
300,0.819800,0.727744
400,0.694400,0.714785
500,0.728100,0.707230
600,0.678800,0.698887
700,0.701700,0.691819
800,0.632200,0.686013
900,0.640000,0.684088
1000,0.609000,0.679561


TrainOutput(global_step=1620, training_loss=0.7473797127052589, metrics={'train_runtime': 393.2874, 'train_samples_per_second': 8.238, 'train_steps_per_second': 4.119, 'total_flos': 211650367979520.0, 'train_loss': 0.7473797127052589, 'epoch': 2.0})

Save Model


In [18]:
trainer.save_model("./victorian_chatbot")
tokenizer.save_pretrained("./victorian_chatbot")


('./victorian_chatbot/tokenizer_config.json',
 './victorian_chatbot/special_tokens_map.json',
 './victorian_chatbot/vocab.json',
 './victorian_chatbot/merges.txt',
 './victorian_chatbot/added_tokens.json',
 './victorian_chatbot/tokenizer.json')

ChatBot

In [19]:
from transformers import pipeline

chatbot = pipeline("text-generation", model="./victorian_chatbot", tokenizer=tokenizer)

def ask_bot(question):
    prompt = f"User: {question}\nBot:"
    response = chatbot(prompt, max_length=150, do_sample=True, top_p=0.95, temperature=0.7)[0]['generated_text']
    print(response.split("Bot:")[-1].strip())

ask_bot("How are you, dear friend?")
ask_bot("What do you think of love?")


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Well, I know, you are not an orphan.
I do not think you have any idea.


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_path = "./victorian_chatbot"  # path to your fine-tuned model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

device = 0 if torch.cuda.is_available() else -1

chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)


Device set to use cuda:0


In [23]:
def ask_bot(question):
    prompt = f"User: {question}\nBot:"
    response = chatbot(
        prompt,
        truncation=True,
        max_length=512,       # max input length
        max_new_tokens=150,   # only generated tokens
        do_sample=True,
        top_p=0.95,
        temperature=0.7
    )[0]['generated_text']

    return response.split("Bot:")[-1].strip()


In [24]:
print("🎩 Victorian Chatbot — type 'exit' to quit.")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break
    bot_reply = ask_bot(user_input)
    print(f"Bot: {bot_reply}\n")


🎩 Victorian Chatbot — type 'exit' to quit.
You: hi


Both `max_new_tokens` (=150) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: You were so very kind.” “You would have done a great service,” said the gentleman.

You: exit
👋 Goodbye!


**General Purpose Document Q/A Chatbot**

In [33]:
!pip install -q langchain transformers sentence-transformers pypdf python-docx openpyxl langchain_community langchain_huggingface langchain_chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 57.8 MB/s eta 0:0

In [4]:
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader, UnstructuredExcelLoader

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files
import os

In [44]:
# -------------------------------
# Load instruction-tuned model
# -------------------------------
model_name = "google/flan-t5-large"   # try "flan-t5-large" if GPU allows
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    temperature=0.7,
    top_p=0.9
)

from transformers import pipeline

# Add summarizer
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", device=0)


def summarize_docs(docs, query="Summarize the story."):
    # Combine retrieved text
    context = " ".join([d.page_content for d in docs[:5]])  # first 5 chunks
    summary = summarizer(context, max_length=200, min_length=80, do_sample=False)[0]['summary_text']
    return summary

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0
Device set to use cuda:0


In [45]:
# -------------------------------
# Custom prompt for clean answers
# -------------------------------
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a polite, knowledgeable assistant.
Answer the user’s question using the context provided.
If the context does not contain the answer, say you don’t know.

Context:
{context}

Question: {question}
Answer:"""
)


In [46]:
# -------------------------------
# Document ingestion function
# -------------------------------
from langchain_core.documents import Document # Import Document class

def load_documents(file_paths, free_text=""):
    docs = []
    for path in file_paths:
        if path.endswith(".pdf"):
            loader = PyPDFLoader(path)
            docs.extend(loader.load())
        elif path.endswith(".docx"):
            loader = Docx2txtLoader(path)
            docs.extend(loader.load())
        elif path.endswith(".xlsx"):
            loader = UnstructuredExcelLoader(path)
            docs.extend(loader.load())
        elif path.endswith(".txt"):
            loader = TextLoader(path)
            docs.extend(loader.load())
    if free_text.strip():
        # Create a Document object for free text
        docs.append(Document(page_content=free_text, metadata={"source": "free_text"}))
    return docs

In [47]:
# -------------------------------
# Build Vectorstore
# -------------------------------
def build_retriever(docs):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(docs, embeddings)
    return vectorstore.as_retriever()

In [48]:
# -------------------------------
# Chatbot creation
# -------------------------------
def create_qa_bot(docs):
    retriever = build_retriever(docs)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": custom_prompt}
    )
    return qa_chain

In [55]:
# -------------------------------
# User Input Section
# -------------------------------
print("📂 Do you want to upload files or just provide free text?")
choice = input("Type 'files' or 'text': ").strip().lower()

file_paths = []
free_text = ""

if choice == "files":
    print("📤 Please upload your documents (PDF, DOCX, XLSX, TXT)...")
    uploaded = files.upload()
    for filename in uploaded.keys():
        file_paths.append(filename)
    print(f"✅ Uploaded: {file_paths}")

elif choice == "text":
    free_text = input("✍️ Enter your text content: ")

else:
    print("⚠️ Invalid choice. Defaulting to free text mode.")
    free_text = input("✍️ Enter your text content: ")

📂 Do you want to upload files or just provide free text?
Type 'files' or 'text': text
✍️ Enter your text content: “Ballerina” is a halfway decent action movie that will suffer because it lives in the massive shadow of John Wick, one of the best modern franchises. It struggles to escape the spectacular, no-misses Keanu Reeves quadrilogy, especially when its weaknesses are matched up specifically with the strength of Chad Stahelski’s films. Yet, this young assassin has also learned a thing or two from her mentor. Much like “Mission: Impossible – The Final Reckoning,” the first half of “Ballerina” requires more patience than the second, when action does the talking and even the editing/choreography tightens up. The last third of “Ballerina” is basically one extended, truly goofy action sequence, and it’s the kind of unpretentious fun that one wants from a movie subtitled “From the World of John Wick.”  Allegedly set between “John Wick: Chapter 3 – Parabellum” and “John Wick: Chapter 4”—al

In [53]:
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

def create_qa_bot(docs):
    # Split text into chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = splitter.split_documents(docs)

    # Create embeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(chunks, embeddings)

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # ✅ Define the LLM here so it's always available
    summarizer = pipeline(
        "text2text-generation",
        model="google/flan-t5-small",  # small + safe
        device=-1  # CPU (avoid CUDA errors)
    )
    llm = HuggingFacePipeline(pipeline=summarizer)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="map_reduce",  # better summarization
    )

    return qa_chain, chunks


In [57]:
print("\n📚 Multi-Input QA Chatbot — type 'exit' to quit.\nType 'summary' to get a summary of the uploaded documents.\n")

docs = load_documents(file_paths=file_paths, free_text=free_text)
qa_chain, chunks = create_qa_bot(docs)

while True:
    query = input("You: ")
    if query.lower() in ["exit", "quit"]:
        print("Bot: Farewell! 📜")
        break
    response = qa_chain.run(query)
    print("Bot:", response)



📚 Multi-Input QA Chatbot — type 'exit' to quit.
Type 'summary' to get a summary of the uploaded documents.



Device set to use cpu


You: summary


Token indices sequence length is longer than the specified maximum sequence length for this model (799 > 512). Running this sequence through the model will result in indexing errors


Bot: The final third of “Ballerina” is a sequel to “Mission: Impossible – The Final Reckoning,” directed by Chad Stahelski.
You: exit
Bot: Farewell! 📜
